# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Task type: Clustering.**

Lane 3 (Structured Content Archetype Clustering) asks *"what kinds of pages exist in the
inventory?"* — not "will this page decline" (classification) or "which page first" (ranking). There
is no observed yes/no outcome and no priority score to predict; the question is about **structure**,
not a target. That maps directly onto clustering: group ~30k pages by safe 90-day metric/metadata
signals, then have a human name the resulting groups and map each to an action. It stays
unsupervised on purpose — I don't want to bake in a rule-based label (like a fixed "declining"
threshold) before I've even looked at the shape of the data.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd, numpy as np
from pathlib import Path

# Robust path: walk up until we find the starter CSV, regardless of execution cwd.
root = Path.cwd()
while not (root / "data" / "raw" / "content_refresh_anonymized.csv").exists() and root != root.parent:
    root = root.parent
DATA = root / "data" / "raw" / "content_refresh_anonymized.csv"

df = pd.read_csv(DATA)

# Confirm there is no ready-made supervised label in this data for "which archetype" —
# the closest thing (trend_direction/trend_pct) is a DIFFERENT question (past trend, not "kind of
# page") and is flagged in the data dictionary as never-a-feature (it derives from the same
# arithmetic it would be "predicting"). No column here answers "what type of page is this."
label_like_cols = [c for c in df.columns if "label" in c.lower() or "type" in c.lower()]
print("columns with 'label' or 'type' in the name:", label_like_cols)
print("\n-> content_type is a CATEGORY of content (blog, product, ...), not an archetype;")
print("   there is no archetype/persona column anywhere in the schema.")

columns with 'label' or 'type' in the name: ['content_type']

-> content_type is a CATEGORY of content (blog, product, ...), not an archetype;
   there is no archetype/persona column anywhere in the schema.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**There is no target — that's the point of clustering.** Unsupervised methods don't predict a
column that already exists; they group rows by similarity and a human names the groups afterward.
The closest thing to a "proxy" is the **cluster assignment itself** (an integer 0..k-1) — but that
integer only becomes meaningful once I inspect the cluster's median signals and give it a name
(e.g. "champion," "hidden gem"), and only after that can it feed an action.

This matters for honesty: I must **not** manufacture a target by thresholding a signal myself (e.g.
"cluster = high CTR AND low position") — that would just be a fixed rule wearing an ML costume. The
groups have to emerge from the joint structure of several signals at once, which is exactly what a
plain if/else can't do (see §5). And the features that go in are restricted to **safe, observable
90-day metrics + metadata** — never `trend_direction`/`trend_pct` (those are the label-trap columns
for the classification lane, not this one) and never raw text/URLs (this is metric/metadata
clustering, not semantic clustering).

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

leakage_cols = ["trend_direction", "trend_pct"]
print("Columns that exist but are excluded as features (label-trap, per data dictionary):")
for c in leakage_cols:
    print(f"  - {c}: present={c in df.columns}")

print("\nConfirming: these describe the OUTCOME question (is a page declining), a different lane.")
print("Lane 3 never trains on them — they are held out of the feature list entirely.")

Columns that exist but are excluded as features (label-trap, per data dictionary):
  - trend_direction: present=True
  - trend_pct: present=True

Confirming: these describe the OUTCOME question (is a page declining), a different lane.
Lane 3 never trains on them — they are held out of the feature list entirely.


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Primary metric: silhouette score**, backed by a **human sense-check** of the cluster medians —
this is the pairing the skill table gives for clustering, and I already have a number for it (see
w01): an untuned k=5 run scored **silhouette ≈ 0.278** on standardized safe features. That's a real,
positive number today, not a promise. "Good" for Week 2+ means: silhouette that holds up (or
improves) after tuning k and handling missingness, **and** clusters stay stable across random seeds
and a client-holdout split (a cluster that only exists for one client isn't an archetype, it's that
client). Silhouette alone can't be the whole story for an *unsupervised* task — a mathematically
separable cluster that makes no editorial sense is still a failure, which is why the human read of
the centroids is part of the metric, not an afterthought.

I will **not** claim success from cluster count or sizes looking "reasonable" — that's a vibe, not
a metric.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

feat = pd.DataFrame({
    "log_impr":          np.log1p(df["impressions_90d"]),
    "ctr":               df["ctr"],
    "avg_position":      df["avg_position"].replace(0, np.nan),   # 0 == "no position data"
    "engagement_rate":   df["engagement_rate"],
    "days_since_update": df["days_since_last_update"],
    "word_count":        df["word_count"],
}).dropna()

Xs = StandardScaler().fit_transform(feat)
km = KMeans(n_clusters=5, random_state=42, n_init=10).fit(Xs)
sil = silhouette_score(Xs, km.labels_)
print(f"today's baseline: silhouette = {sil:.3f} on {len(feat):,} pages (k=5, untuned)")
print("this is the number Week 2+ tuning has to beat, not a final result.")

today's baseline: silhouette = 0.278 on 21,109 pages (k=5, untuned)
this is the number Week 2+ tuning has to beat, not a final result.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**One row = one content item (page)**, described by its pseudonymized IDs, metadata (content type,
age, freshness), and safe 90-day observable signals (impressions, CTR, position, engagement, scroll,
AI traffic). `client_id` exists only for grouping/holdout splits, never as a feature.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

unit_of_analysis = df[[
    "content_id", "client_id", "content_type", "age_tier", "freshness_tier",
    "impressions_90d", "ctr", "avg_position", "engagement_rate", "word_count",
]]
print(f"one row = one page  |  {len(unit_of_analysis):,} rows  |  {unit_of_analysis['content_id'].nunique():,} unique content_id (no dupes)")
unit_of_analysis.sample(5)

one row = one page  |  30,000 rows  |  30,000 unique content_id (no dupes)


,content_id,client_id,content_type,age_tier,freshness_tier,impressions_90d,ctr,avg_position,engagement_rate,word_count
23687,content_7850ca78a43a,client_d4735e3a26,feedly article,181-365,0-30,1,0.00,0.0,0.0,838.0
4127,content_d92e06dd3ce7,client_349c41201b,keyword article,91-180,0-30,593,0.67,2.5,0.0,2899.0
9328,content_de5b0089c9f3,client_d4735e3a26,feedly article,181-365,0-30,1,0.00,0.0,0.0,882.0
3929,content_6ab1ba993c43,client_f369cb89fc,keyword article,91-180,0-30,51,1.96,4.7,0.0,2560.0
29438,content_3d3ce5a7f0af,client_19581e27de,keyword article,365+,0-30,284,0.00,54.7,0.0,NaN


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule (e.g. "if impressions > X and CTR < Y then archetype = leaking-traffic") only works
along axes a human picked in advance, and it breaks the moment two signals move together in an
unexpected way. The check below shows the candidate signals are only **weakly correlated** with
each other — pages vary along several *partly-independent* axes at once (visibility, position, CTR,
engagement, freshness, length). With 6 nearly-independent axes there isn't a small number of
if/else branches that captures the joint structure; the archetypes live in the interaction between
signals, which is exactly what distance-based clustering finds and a hand-written rule can't
enumerate. A rule also can't be *checked* the way silhouette can — clustering gives me a number to
argue about, a rule just gives me my own assumptions back.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

corr = feat.corr(numeric_only=True).round(2)
print("Pairwise correlation of candidate signals (mostly weak -> several independent axes):")
print(corr.to_string())

max_off_diag = corr.where(~np.eye(len(corr), dtype=bool)).abs().max().max()
print(f"\nstrongest pairwise correlation among candidate signals: {max_off_diag:.2f}")
print("-> no pair is redundant enough to collapse into a single if-statement axis.")

Pairwise correlation of candidate signals (mostly weak -> several independent axes):
                   log_impr   ctr  avg_position  engagement_rate  days_since_update  word_count
log_impr               1.00 -0.15          0.01             0.07               0.13        0.33
ctr                   -0.15  1.00         -0.08             0.11              -0.03       -0.13
avg_position           0.01 -0.08          1.00            -0.03               0.15        0.10
engagement_rate        0.07  0.11         -0.03             1.00              -0.04       -0.07
days_since_update      0.13 -0.03          0.15            -0.04               1.00        0.32
word_count             0.33 -0.13          0.10            -0.07               0.32        1.00

strongest pairwise correlation among candidate signals: 0.33
-> no pair is redundant enough to collapse into a single if-statement axis.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.